# 01 — Build Embeddings & FAISS Index

This notebook is the **foundation** of the SpatialSearch pipeline. It runs once and saves everything to disk so the other notebooks don't need to re-encode 110k tweets from scratch every time.

**What happens here:**
1. Load all tweets from `tweets-utf-8.json`
2. Encode with GloVe (static word embeddings) and MiniLM (contextual BERT-based embeddings)
3. Build a FAISS index for fast nearest-neighbor search
4. Save embeddings and index to disk

**Run this before any other notebook.**

In [1]:
# --- STEP 0: INSTALL DEPENDENCIES ---
# faiss-cpu provides the vector index; sentence-transformers wraps both GloVe and MiniLM
!pip install faiss-cpu sentence-transformers --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
yfinance 0.2.65 requires websockets>=13.0, but you have websockets 10.4 which is incompatible.
numba 0.55.1 requires numpy<1.22,>=1.18, but you have numpy 2.0.2 which is incompatible.


In [2]:
# --- STEP 1: IMPORTS ---
import json
import os
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

print("All imports successful.")

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
# --- STEP 2: LOAD TWEETS ---

def load_tweets(filepath):
    """
    Load tweet texts from a JSON file where each line is a separate JSON object.

    Parameters:
        filepath (str): Path to the tweets JSON file.

    Returns:
        list[str]: A list of tweet text strings, skipping any entries
                   that are missing a 'text' field or can't be parsed.
    """
    tweets = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                # Some entries may be metadata or malformed — only keep actual tweet text
                if 'text' in obj:
                    tweets.append(obj['text'])
            except json.JSONDecodeError:
                pass  # skip malformed lines silently
    return tweets


# Try Colab path first, then fall back to local directory
data_path = '/content/tweets-utf-8.json' if os.path.exists('/content/tweets-utf-8.json') else 'tweets-utf-8.json'

tweets = load_tweets(data_path)
print(f"Loaded {len(tweets):,} tweets from {data_path}")
print(f"Sample tweet: {tweets[0][:100]}...")

In [ ]:
# --- STEP 3: SAVE CLEAN TWEET LIST ---
# Save tweet texts as a plain JSON list so other notebooks can load them
# without re-parsing the original file (which has metadata we don't need)

with open('tweets.json', 'w', encoding='utf-8') as f:
    json.dump(tweets, f, ensure_ascii=False)

print(f"Saved {len(tweets):,} tweet texts to tweets.json")

In [ ]:
# --- STEP 4: ENCODE WITH GLOVE ---
# GloVe gives each word a fixed vector regardless of context.
# SentenceTransformers averages word vectors to produce a sentence embedding.
# This is fast but can't capture word sense (e.g. "bank" the river vs. bank the institution).

def encode_tweets(model_name, tweets, batch_size=256, show_progress=True):
    """
    Load a SentenceTransformer model and encode a list of tweets.

    Parameters:
        model_name (str): HuggingFace model identifier.
        tweets (list[str]): Tweet texts to encode.
        batch_size (int): How many tweets to encode at once. Larger = faster but more RAM.
        show_progress (bool): Whether to show a tqdm progress bar.

    Returns:
        np.ndarray: Float32 array of shape (num_tweets, embedding_dim).
    """
    model = SentenceTransformer(model_name)
    embeddings = model.encode(
        tweets,
        batch_size=batch_size,
        show_progress_bar=show_progress,
        convert_to_numpy=True
    )
    # Ensure float32 — FAISS requires this dtype
    return embeddings.astype(np.float32)


print("Encoding with GloVe (average_word_embeddings_glove.840B.300d)...")
print("This encodes 110k tweets using averaged GloVe word vectors — expect a few minutes.")
glove_embeddings = encode_tweets('average_word_embeddings_glove.840B.300d', tweets)

np.save('glove_embeddings.npy', glove_embeddings)
print(f"GloVe embeddings shape: {glove_embeddings.shape}")
print(f"Saved to glove_embeddings.npy")

In [ ]:
# --- STEP 5: ENCODE WITH MINILM ---
# MiniLM is a distilled BERT model — it reads the full sentence at once and
# produces context-aware embeddings. Much better at capturing meaning than GloVe.
# The tradeoff is it's slower to encode, but results are significantly better.

import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

print("Encoding with MiniLM (all-MiniLM-L6-v2)...")
print("MiniLM produces 384-dim contextual embeddings — slower than GloVe but much richer.")
minilm_embeddings = encode_tweets('all-MiniLM-L6-v2', tweets)

np.save('minilm_embeddings.npy', minilm_embeddings)
print(f"MiniLM embeddings shape: {minilm_embeddings.shape}")
print(f"Saved to minilm_embeddings.npy")

In [ ]:
# --- STEP 6: BUILD FAISS INDEX FOR MINILM ---

def build_inner_product_index(embeddings):
    """
    Build a FAISS IndexFlatIP (inner product) index from embeddings.

    Inner product on L2-normalized vectors is equivalent to cosine similarity,
    so normalizing first lets us use the fast IP index to rank by cosine sim.

    Parameters:
        embeddings (np.ndarray): Float32 array of shape (n, dim).

    Returns:
        faiss.IndexFlatIP: Populated FAISS index ready to query.
    """
    # Normalize so dot product equals cosine similarity — required for IndexFlatIP
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    # Avoid division by zero for any zero vectors
    norms = np.where(norms == 0, 1, norms)
    normalized = (embeddings / norms).astype(np.float32)

    dim = normalized.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(normalized)
    return index


print("Building FAISS IndexFlatIP for MiniLM embeddings...")
# Why FAISS over brute-force NumPy: FAISS uses optimized C++ under the hood
# and can scale to millions of vectors. Our cosine search in the original notebook
# was O(n) — FAISS is still exact here (IndexFlatIP) but much faster in practice.
minilm_index = build_inner_product_index(minilm_embeddings)

faiss.write_index(minilm_index, 'minilm_faiss.index')
print(f"Indexed {minilm_index.ntotal:,} MiniLM vectors — saved to minilm_faiss.index")

In [ ]:
# --- STEP 7: BUILD FAISS INDEX FOR GLOVE ---

def build_l2_index(embeddings):
    """
    Build a FAISS IndexFlatL2 (Euclidean distance) index from embeddings.

    We use L2 for GloVe because GloVe vectors aren't trained to have unit norm,
    so inner product scores are less meaningful without careful normalization.
    L2 distance is more robust here.

    Parameters:
        embeddings (np.ndarray): Float32 array of shape (n, dim).

    Returns:
        faiss.IndexFlatL2: Populated FAISS index ready to query.
    """
    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings.astype(np.float32))
    return index


print("Building FAISS IndexFlatL2 for GloVe embeddings...")
glove_index = build_l2_index(glove_embeddings)

faiss.write_index(glove_index, 'glove_faiss.index')
print(f"Indexed {glove_index.ntotal:,} GloVe vectors — saved to glove_faiss.index")

In [ ]:
# --- STEP 8: SMOKE TEST ---
# Quick sanity check: run a query through the MiniLM FAISS index
# and make sure the returned tweets look semantically relevant.

from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
test_query = "I am looking for a job."

query_vec = model.encode([test_query], convert_to_numpy=True).astype(np.float32)
# Normalize query vector the same way we normalized the index
query_vec = query_vec / np.linalg.norm(query_vec)

# Search for top 5 most similar tweets
scores, indices = minilm_index.search(query_vec, k=5)

print(f"Test query: '{test_query}'")
print(f"\nTop 5 results (MiniLM FAISS):")
for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
    print(f"  {rank}. [score={score:.4f}] {tweets[idx][:120]}")

print("\nSmoke test passed — index is working correctly!")

In [ ]:
# --- SUMMARY ---
print("=" * 60)
print("INDEX BUILD COMPLETE")
print("=" * 60)
print(f"  tweets.json           — {len(tweets):,} tweet texts")
print(f"  glove_embeddings.npy  — shape {glove_embeddings.shape}")
print(f"  minilm_embeddings.npy — shape {minilm_embeddings.shape}")
print(f"  glove_faiss.index     — {glove_index.ntotal:,} vectors (L2)")
print(f"  minilm_faiss.index    — {minilm_index.ntotal:,} vectors (IP/cosine)")
print()
print("Next: run 02_retrieval_pipeline.ipynb")